In [ ]:
# This mounts your Google Drive to the Colab VM.
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# TODO: Enter the foldername in your Drive where you have saved the unzipped
# assignment folder, e.g. 'cs231n/assignments/assignment3/'
FOLDERNAME = "cs231n/assignments/assignment3/"
assert FOLDERNAME is not None, "[!] Enter the foldername."

# Now that we've mounted your Drive, this ensures that
# the Python interpreter of the Colab VM can load
# python files from within it.
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

In [ ]:
# This downloads the COCO dataset to your Drive if it doesn't already exist
# (you should already have this dataset from a previous notebook!)
# Uncomment the following if you don't have it.
# %cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/datasets/
# !bash get_coco_captioning.sh
# %cd /content/drive/My\ Drive/$FOLDERNAME

In [ ]:
# Some useful python libraries
! pip install ftfy regex tqdm
! pip install git+https://github.com/openai/CLIP.git
! pip install decord

# State-of-the-Art Pretrained Image Models

In the previous exercise, you learned about [SimCLR](https://arxiv.org/abs/2002.05709) and how contrastive self-supervised learning can be used to learn meaningful image representations. In this notebook, we will explore two more recent models that also aim to learn high-quality visual representations and have demonstrated strong and robust performance on a variety of downstream tasks.


First, we will examine the [CLIP](https://github.com/openai/CLIP) model. Like SimCLR, CLIP uses a contrastive learning objective, but instead of contrasting two augmented views of the same image, it contrasts two different modalities: text and image. To train CLIP, OpenAI collected a large dataset of ~400M image-text pairs from the internet, including sources like Wikipedia and image alt text. The resulting model learns rich, high-level image features and has achieved impressive zero-shot performance on many vision benchmarks.

Next, we will explore [DINO](https://github.com/facebookresearch/dino), a self-supervised learning method for vision tasks that applies contrastive learning in a self-distillation framework with multi-crop augmentation strategy. The authors showed that the features learned by DINO ViTs are fine-grained and semantically rich with explicit information about the semantic segmentation of the image.




# CLIP

As explained above, CLIP's training objective incorporates both text and images, building upon the principles of contrastive learning. Consider this quote from the SimCLR notebook:
>The goal of the contrastive loss is to maximize agreement between the final vectors **$z_i = g(h_i)$** and **$z_j = g(h_j)$**.

Similarly, CLIP is trained to maximize agreement between two vectors. However, because these vectors come from different modalities, CLIP uses two separate encoders: a transformer-based Text Encoder and a Vision Transformer (ViT)-based Image Encoder. Note that some smaller, more efficient versions of CLIP use a ResNet as the Image Encoder instead of a ViT.

Run the cell below to visualize the training and inference pipeline of CLIP.

During the pretraining phase, each batch consists of multiple images along with their corresponding captions. Each image is independently processed by an Image Encoder—typically a visual model like a Vision Transformer (ViT) or a Convolutional Neural Network (ConvNet)—which produces an image embedding $I_n$. Likewise, each caption is independently processed by a Text Encoder to generate a corresponding text embedding $T_n$. Next, we compute the pairwise similarities between all image-text combinations, meaning each image is compared with every caption, and vice versa. The training objective is to maximize the similarity scores along the diagonal of the resulting similarity matrix -- that is, the scores for the matching image-caption pairs $(I_n, T_n)$.  Through backpropagation, the model learns to assign higher similarity scores to true matches than to mismatched pairs.

Through this setup, CLIP effectively learns to represent images and texts in a shared latent space. In this space, semantic concepts are encoded in a modality-independent way, enabling meaningful cross-modal comparisons between visual and textual inputs.



In [ ]:
from IPython.display import Image as ColabImage
ColabImage(f'/content/drive/My Drive/{FOLDERNAME}/CLIP.png')

**Inline Question 1** -

Why does CLIP's learning depend on the batch size? If the batch size is fixed, what strategy can we use to learn rich image features?

$\color{blue}{\textit Your Answer:}$


**为什么 CLIP 的学习依赖于 batch size？**

CLIP 使用对比学习目标：对于 batch 中的每张图像，它需要将其与对应的文本描述（正样本对）匹配，同时将其与 batch 中所有**其他**文本描述（负样本）区分开来。

具体来说，对于一个大小为 $N$ 的 batch，CLIP 会构建一个 $N \times N$ 的相似度矩阵，每个样本有 $1$ 个正样本对和 $N-1$ 个负样本对。因此：

- **batch size 越大** → 负样本越多 → 对比学习的难度越高 → 模型被迫学习更具区分性的特征表示
- **batch size 越小** → 负样本越少 → 对比信号弱 → 模型难以学到丰富的语义特征

这也是为什么原始 CLIP 论文使用了极大的 batch size（**32,768**），这对于普通硬件来说几乎不可行。

---

**若 batch size 固定，可以使用什么策略？**

可以使用 **Memory Queue（记忆队列）** 策略，即 **MoCo（Momentum Contrast）** 中提出的方法：

- 维护一个动态队列，存储过去多个 batch 的编码特征作为负样本库
- 队列容量可以远大于单个 batch size，从而提供大量负样本
- 使用一个**动量编码器（Momentum Encoder）** 来稳定地更新队列中的特征，避免特征表示不一致的问题

$$\theta_k \leftarrow m \cdot \theta_k + (1 - m) \cdot \theta_q$$

这样即使 batch size 固定且较小，模型仍然能接触到大量多样化的负样本，从而学到丰富的图像特征表示。



# Loading COCO dataset

We'll use the same captioning dataset you used to train your RNN captioning model, but instead of generating the captions lets see if we can match each image to the correct caption.

In [ ]:
%load_ext autoreload
%autoreload 2

import time, os, json
import numpy as np
import matplotlib.pyplot as plt
import torch
import clip
import torch
from tqdm.auto import tqdm

from PIL import Image
from cs231n.clip_dino import *

def rel_error(x, y):
    """Returns relative error."""
    return np.max(np.abs(x - y) / (np.maximum(1e-10, np.abs(x) + np.abs(y))))


In [ ]:
from cs231n.coco_utils import load_coco_data, sample_coco_minibatch, decode_captions
from cs231n.image_utils import image_from_url

In [ ]:
# Load COCO data from disk into a dictionary.
# this is the same dataset you used for the RNN captioning notebook :)
data = load_coco_data(pca_features=True)

# Print out all the keys and values from the data dictionary.
for k, v in data.items():
    if type(v) == np.ndarray:
        print(k, type(v), v.shape, v.dtype)
    else:
        print(k, type(v), len(v))

In [ ]:
# we're just using the loaded captions from COCO, so we need to decode them and get rid of the special tokens.
decoded_captions= []
for caption in data['val_captions']:
  caption = decode_captions(caption, data['idx_to_word'])\
    .replace('<START>', '')\
    .replace('<END>', '')\
    .replace('<UNK>', '')\
    .strip()
  decoded_captions.append(caption)

In [ ]:
# lets get 10 examples
mask = np.array([135428, 122586, 122814, 133173, 176639, 163828,  98169,   6931,
        19488, 175760])
first_captions = [decoded_captions[elem] for elem in mask]

img_idxs = data['val_image_idxs'][mask]       # the images the captions refer to
first_images   = [image_from_url(data['val_urls'][j]) for j in img_idxs]

In [ ]:
for i, (caption, image) in enumerate(zip(first_captions, first_images)):
    plt.imshow(image)
    plt.axis('off')
    caption_str = caption
    plt.title(caption_str)
    plt.show()

# Running the CLIP Model

First we'll use the pretrained CLIP model to extract features from the texts and images separetely.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)

In [ ]:
# You can check the model layers by printing the model.
# CLIP's model code is available at https://github.com/openai/CLIP/tree/main/clip
# print(clip_model)

In [ ]:
# First, we encode the captions into vectors in the shared embedding space.
# Since we're using a Transformer as the text encoder, we need to tokenize the text first.
text_tokens = clip.tokenize(first_captions).to(device)
with torch.no_grad():
    text_features = clip_model.encode_text(text_tokens)

# Sanity check, print the shape
print(text_features.shape)

In [ ]:
# Then, we encode the images into the same embedding space.
processed_images = [
    clip_preprocess(Image.fromarray(img)).unsqueeze(0)
    for img in first_images
]
images_tensor = torch.cat(processed_images, dim=0).to(device)

with torch.no_grad():
    image_features = clip_model.encode_image(images_tensor)

# sanity check, print the shape
print(image_features.shape)

Open `cs231n/clip_dino.py` and implement `get_similarity_no_loop` to compute similarity scores between text features and image features. Test your implementation below, you should see relative errors less than 1e-5.

In [ ]:
from cs231n.clip_dino import get_similarity_no_loop
torch.manual_seed(231)
np.random.seed(231)
M, N, D = 5, 6, 10

test_text_features = torch.randn(N, D)
test_image_features = torch.randn(M, D)
out = get_similarity_no_loop(test_text_features, test_image_features)

expected_out = np.array([
    [ 0.1867811 , -0.23494351,  0.44155994, -0.18950461,  0.00100103],
    [ 0.17905031, -0.25469488, -0.64330417,  0.25097957, -0.09327742],
    [-0.4407011 , -0.4365381 ,  0.32857686, -0.3765278 ,  0.01049389],
    [ 0.24815483,  0.42157224, -0.08459304,  0.14132318, -0.26935193],
    [ 0.02309848, -0.01441101,  0.5469337 ,  0.6018773 ,  0.21581158],
    [ 0.41579214, -0.014449  , -0.7242257 ,  0.39348006,  0.0822239 ],
]).astype(np.float32)

print("relative error: ", rel_error(out.numpy(), expected_out))

In [ ]:
# Let's visualize the similarities between our batch of images and their captions.

similarities = get_similarity_no_loop(text_features, image_features).cpu().detach().numpy()

plt.figure(figsize=(20, 14))
plt.imshow(similarities, vmin=0.1, vmax=0.3)
plt.yticks(range(len(text_features)), first_captions, fontsize=18)
plt.xticks([])
for i, image in enumerate(first_images):
    plt.imshow(image, extent=(i - 0.5, i + 0.5, -1.6, -0.6), origin="lower")
for x in range(similarities.shape[1]):
    for y in range(similarities.shape[0]):
        plt.text(x, y, f"{similarities[y, x]:.2f}", ha="center", va="center", size=12)

for side in ["left", "top", "right", "bottom"]:
    plt.gca().spines[side].set_visible(False)

plt.xlim([-0.5, len(image_features) - 0.5])
plt.ylim([len(text_features) + 0.5, -2])

plt.title("Cosine similarity between text and image features", size=20)
plt.show()

# Zero Shot Classifier

You will be able to see a high similarity between matching image-caption pairs above. We can leverage this property to design an image classifier that doesn't require any labeled data (i.e., a zero-shot classifier). Each class can be represented using an appropriate natural language description, and any input image will be classified into the class whose description has the highest similarity with the image in CLIP's embedding space.

Implement `clip_zero_shot_classifier` in `cs231n/clip_dino.py` and test it below. You should be able to see the following predictions:

['a person', 'an animal', 'an animal', 'food', 'a person', 'a landscape', 'other', 'other', 'other', 'a person']

In [ ]:
from cs231n.clip_dino import clip_zero_shot_classifier

classes = ["a person", "an animal", "food", "a landscape", "other"]

pred_classes = clip_zero_shot_classifier(
    clip_model, clip_preprocess, first_images, classes, device)

print(pred_classes)

Run the cell below to visualize the predictions. As you can see, CLIP offers a straightforward way to perform reasonable zero-shot classification across any class taxonomy.

CLIP was the first model to outperform standard supervised training on ImageNet classification without using any ImageNet images or labels (The original CLIP paper has many such interesting experiments and analysis).


In [ ]:
# Visualize the zero shot predictions
for i, (pred_class, image) in enumerate(zip(pred_classes, first_images)):
    plt.imshow(image)
    plt.axis('off')
    plt.title(pred_class)
    plt.show()

# Image Retrieval using CLIP

Just as we used CLIP to retrieve the matching class name for each image, we can also use it to retrieve matching images from text inputs (semantic image retrieval). Implement the `CLIPImageRetriever` in `cs231n/clip_dino.py` and test it by running the two cells below. The expected top 2 outputs for each query are provided in the comments.

In [ ]:
from cs231n.clip_dino import CLIPImageRetriever
clip_retriever = CLIPImageRetriever(clip_model, clip_preprocess, first_images, device)

In [ ]:
query = "sports"  # tennis, skateboard
# query = "black and white"  # bathroom, zerbas
img_indices = clip_retriever.retrieve(query)

for img_index in img_indices:
    plt.imshow(first_images[img_index])
    plt.axis('off')
    plt.show()

**Inline Question 2** -

CLIP learns to align image and text representations in a shared latent space using a contrastive loss. How would you extend this idea to more than two modalities?

$\color{blue}{\textit Your Answer:}$

## 回答

**将 CLIP 扩展到多模态的核心思路**

CLIP 的本质是将两种模态（图像 + 文本）映射到同一个共享的潜在空间，使正样本对的表示相互靠近，负样本对相互远离。将这一思想推广到三种及以上模态（如图像、文本、音频、视频、深度图等）有以下几种策略：

---

### 方法一：共享潜在空间（All-to-One Alignment）

为每种模态设计独立的编码器，但将所有模态的输出都映射到**同一个公共嵌入空间**：

$$f_{\text{image}}(x), \quad f_{\text{text}}(x), \quad f_{\text{audio}}(x), \quad \cdots \rightarrow \mathbb{R}^d$$

对比损失在所有模态对之间成对计算：

$$\mathcal{L} = \sum_{i < j} \mathcal{L}_{\text{contrastive}}(\text{modality}_i, \text{modality}_j)$$

- **优点**：结构简单，任意两个模态之间可以直接检索
- **缺点**：模态数量增多时，训练对数量以 $O(M^2)$ 增长，计算开销大

---

### 方法二：以某一模态为锚点（Anchor-based Alignment）

选取一种模态（通常是**文本**）作为中心锚点，其他模态都与锚点对齐，而不要求模态间两两对齐：

$$\mathcal{L} = \mathcal{L}(f_{\text{text}}, f_{\text{image}}) + \mathcal{L}(f_{\text{text}}, f_{\text{audio}}) + \mathcal{L}(f_{\text{text}}, f_{\text{video}}) + \cdots$$

文本天然适合作为锚点，因为它能描述几乎所有模态的语义内容。ImageBind 就采用了类似的思路，以图像为锚点对齐六种模态。

- **优点**：训练对数量线性增长 $O(M)$，效率高
- **缺点**：非锚点模态之间的对齐是**隐式的**，依赖于锚点模态的传递性

---

### 方法三：多模态对比损失（Generalized N-way Contrastive Loss）

将 CLIP 的二元对比损失推广为 $M$ 路联合对比损失。对于一个包含 $M$ 种模态的样本组合 $(m_1, m_2, \ldots, m_M)$，定义联合相似度并对所有模态同时对齐：

$$\mathcal{L} = -\log \frac{\exp\left(\sum_{i<j} \text{sim}(z_i, z_j) / \tau\right)}{\sum_{\text{neg}} \exp\left(\sum_{i<j} \text{sim}(z_i, z_j^-) / \tau\right)}$$

- **优点**：能捕捉多模态之间的高阶语义关联
- **缺点**：需要严格的多模态配对数据，现实中难以大规模获取

---

### 总结对比

| 策略 | 对齐方式 | 计算复杂度 | 跨模态检索 |
|---|---|---|---|
| 两两对齐 | 所有模态对 | $O(M^2)$ | 直接 |
| 锚点对齐 | 所有模态 → 锚点 | $O(M)$ | 经由锚点中转 |
| 联合对比 | 多路同时对齐 | $O(M^2)$+ | 最强，但数据要求高 |

在实践中，**锚点对齐（如 ImageBind）** 是目前扩展到多模态最实用的方案，因为它在训练效率和对齐质量之间取得了良好的平衡，同时利用了模态对齐的**传递性**：若模态 A 和 B 都与文本对齐，则 A 和 B 在嵌入空间中也会隐式地相互靠近。

# DINO

As mentioned earlier, models trained with vanilla contrastive learning methods such as SimCLR and CLIP require very large batch sizes. This makes them computationally expensive and limits their accessibility. Subsequent works, like [BYOL](https://arxiv.org/abs/2006.07733), propose an alternative approach that avoids the need for numerous negative samples by using a student-teacher framework. This method performs surprisingly well and was later adopted by [DINO](https://arxiv.org/abs/2104.14294) .

Similar to SimCLR, DINO is trained to maximize the agreement between two vectors derived from different views of the same image. However, unlike SimCLR, DINO uses two separate encoders which are trained differently. The student network is updated via backpropagation to match the outputs of the teacher network. The teacher network is not updated via backpropagation; instead, its weights are updated using an exponential moving average (EMA) of the student's weights. This means that the teacher model evolves more slowly and provides a stable target for the student to learn from.

Run the cell below to visualize the DINO training pipeline.

In [ ]:
from IPython.display import Image as ColabImage
ColabImage(f'/content/drive/My Drive/{FOLDERNAME}/dino.gif')

In [ ]:
# first let's get rid of the CLIP model that's currently using memory
del clip_model
# Uncomment the following if you are using GPU runtime
# torch.cuda.empty_cache()
# torch.cuda.ipc_collect()

In [ ]:
# Load smallest dino model. ViT-S/8. Here ViT-S has ~22M parameters and
# works on 8x8 patches.
dino_model = torch.hub.load('facebookresearch/dino:main', 'dino_vits8')
dino_model.eval().to(device)


In [ ]:
# the image we will be playing around with
sample_image = Image.fromarray(first_images[0]).convert("RGB")
sample_image

# DINO Attention Maps

Since the loaded DINO checkpoint is based on the ViT architecture, we can visualize what each attention head is focusing on. The code below generates heatmaps showing which patches of the original image the [CLS] token attends to across the various heads in the final layer. Although this model was trained using a self-supervised objective without any explicit instruction to recognize "structure" in images, still...

Do you notice any patterns?

In [ ]:
# Preprocess
from torchvision import transforms as T
transform = T.Compose([
    T.Resize((480, 480)),
    T.ToTensor(),
    T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])
img_tensor = transform(sample_image)
w, h = img_tensor.shape[1:]
img_tensor = img_tensor[None].to(device)

# Extract attention
with torch.no_grad():
    attn = dino_model.get_last_selfattention(img_tensor)[0, :, 0, 1:]
nh, tokens = attn.shape
w_feat, h_feat = w // 8, h // 8
attn = attn.reshape(nh, w_feat, h_feat)
attn = torch.nn.functional.interpolate(attn.unsqueeze(0), scale_factor=8, mode="nearest")[0].cpu().numpy()

# Plot attention heads
fig, axes = plt.subplots(1, nh, figsize=(3 * nh, 3))
for i in range(nh):
    ax = axes[i] if nh > 1 else axes
    ax.imshow(attn[i], cmap='inferno')
    ax.axis('off')
plt.show()

In [ ]:
# Extract patch token features and discard [CLS] token.
with torch.no_grad():
    all_tokens = dino_model.get_intermediate_layers(img_tensor, n=1)[0]  # (1, 1+N, D)
    patch_tokens = all_tokens[:, 1:, :]  # (N, D)

print(img_tensor.shape)
print(all_tokens.shape)
print(patch_tokens.shape)


**Inline Question 3**

How do we get the tensor shapes printed above? Explain your answer.


$\color{blue}{\textit Your Answer:}$



---

## 各张量形状推导

### 输入张量
| 张量 | 形状 | 说明 |
|---|---|---|
| `out_left` | $(N, D)$ | $N$ 个样本，每个样本经过投影头得到 $D$ 维向量 |
| `out_right` | $(N, D)$ | 同一批样本的另一增强视图 |

### 拼接后
```python
out = torch.cat([out_left, out_right], dim=0)  # (2N, D)
```
沿 `dim=0`（行方向）拼接，行数翻倍，列数不变，得到 $(2N, D)$。

---

### 相似度矩阵
```python
out_norm = out / norm(out, dim=1, keepdim=True)  # (2N, D)
sim_matrix = out_norm @ out_norm.T               # (2N, D) x (D, 2N) = (2N, 2N)
```
$(2N, D) \times (D, 2N) \rightarrow (2N, 2N)$，矩阵中每个元素 $(i,j)$ 是第 $i$ 和第 $j$ 个增强样本之间的余弦相似度。

---

### 指数矩阵 + mask
```python
exponential = exp(sim_matrix / tau)  # (2N, 2N)
# 去掉对角线后：
exponential = exponential.masked_select(mask).view(2*N, -1)  # (2N, 2N-1)
```
- 原始：$(2N, 2N)$
- 去掉每行的自身项（对角线）后，每行剩 $2N-1$ 个元素
- reshape 得到 $(2N, 2N-1)$

---

### 分母向量
```python
denom = exponential.sum(dim=1, keepdim=True)  # (2N, 1)
```
对每行的 $2N-1$ 个元素求和，保持维度，得到 $(2N, 1)$。

---

### 正样本对相似度
```python
pos_pairs = sim_positive_pairs(out_left, out_right)      # (N, 1)
pos_pairs = torch.cat([pos_pairs, pos_pairs], dim=0)     # (2N, 1)
```
- `out_left` 和 `out_right` 各 $N$ 行，逐行点积得 $(N, 1)$
- 正样本对 $(k, k+N)$ 和 $(k+N, k)$ 相似度相同，复制拼接得 $(2N, 1)$

---

### 损失标量
```python
numerator = exp(pos_pairs / tau)         # (2N, 1)
loss = -log(numerator / denom).mean()   # scalar
```
$(2N, 1)$ 除以 $(2N, 1)$，取 $-\log$ 后对所有 $2N$ 个元素求均值，最终得到一个**标量**损失值。

---

### 形状演变总结

$$
(N, D) \xrightarrow{\text{cat}} (2N, D) \xrightarrow{\text{matmul}} (2N, 2N) \xrightarrow{\text{mask}} (2N, 2N{-}1) \xrightarrow{\text{sum}} (2N, 1) \xrightarrow{\text{mean}} \text{scalar}
$$



# DINO Features

To understand what the model is encoding in each patch, we can visualize the contents of each patch token. Since these embeddings are high-dimensional and difficult to interpret directly, we'll use PCA to identify the directions of highest variance in the feature space.

In the next cell, we visualize the three principal directions of variance in the feature space. This reveals the dominant structure that the patch embeddings are capturing.

In [ ]:
from sklearn.decomposition import PCA

np.random.seed(231)

# PCA
pca = PCA(n_components=3)
patch_pca = pca.fit_transform(patch_tokens.cpu().numpy()[0])

# Normalize PCA components to [0, 1] for RGB display
patch_rgb = (patch_pca - patch_pca.min(0)) / (patch_pca.max(0) - patch_pca.min(0))

# Reshape to image grid (60x60, 3)
patch_rgb_img = patch_rgb.reshape(60, 60, 3)

# Show as image
plt.figure(figsize=(6, 6))
plt.imshow(patch_rgb_img)
plt.axis('off')
plt.title("Patch Embeddings (PCA → RGB)")
plt.show()

**Inline Question 4** -

What kind of structure do you see in the visualization above? What does it imply when a region consistently appears in a specific color? What does it mean when two regions have distinctly different color? Remember that PCA reveals the directions of highest variance in the feature space across all patches. A patch's color reflects its distinct feature content.


$\color{blue}{\textit Your Answer:}$


### 可视化中呈现的结构

PCA 可视化将图像中每个 patch 的高维特征向量降维到前 3 个主成分，并映射到 RGB 颜色通道上显示。通常可以观察到以下结构：

- **语义区域的颜色聚类**：属于同一语义类别的区域（如背景、前景物体、天空、地面）往往呈现**相似的颜色**，说明模型已经学会将语义相似的 patch 映射到特征空间中相近的位置
- **物体边界处的颜色突变**：不同语义区域之间的边界处颜色发生明显变化，表明模型能够自动捕捉物体的轮廓信息
- **全局一致性**：同一物体的不同 patch 颜色趋于一致，即使它们在图像中位置分散

---

### 某区域颜色一致意味着什么？

若一个区域**持续呈现同一种颜色**，说明该区域内所有 patch 的特征向量在 PCA 主方向上的投影值非常接近，即：

$$\text{PCA}(f_{\text{patch}_i}) \approx \text{PCA}(f_{\text{patch}_j}), \quad \forall\, i, j \in \text{同一区域}$$

这意味着模型将这些 patch **编码为相似的特征表示**，反映出它们具有相同的语义内容或视觉纹理。例如，草地区域的所有 patch 颜色一致，说明模型识别出它们同属一类语义概念。

---

### 两个区域颜色截然不同意味着什么？

若两个区域**颜色差异显著**，说明它们的特征向量在高方差方向上的投影差异很大，即这两个区域在特征空间中**相距较远**：

$$\|\text{PCA}(f_{\text{region}_A}) - \text{PCA}(f_{\text{region}_B})\| \gg 0$$

这意味着模型已经学会**区分不同的语义内容**，例如前景物体与背景之间的颜色差异，说明模型具备了一定的语义分割能力，尽管它从未在分割任务上显式训练过。

---

### 整体含义总结

| 观察现象 | 含义 |
|---|---|
| 同一物体颜色一致 | 模型学到了语义一致的特征表示 |
| 不同物体颜色不同 | 模型能区分不同语义类别 |
| 背景与前景对比鲜明 | 模型隐式学会了前景/背景分离 |
| 颜色在多张图中规律出现 | 该主成分方向捕捉了跨图像的共性语义结构 |

PCA 可视化本质上揭示了**特征空间中方差最大的方向**，颜色差异越大，说明对应 patch 在这些主方向上的特征差异越显著，也就意味着模型赋予了它们**不同的语义身份**。这正是自监督表示学习（如 SimCLR、DINO）能够在无标签数据上涌现出语义理解能力的有力证据。


# A Simple Segmentation Model over DINO Features

In the previous section, we saw that DINO features can provide surprisingly good segmentation cues. Now, let's put that idea to the test by training a simple segmentation model on the [DAVIS dataset](https://davischallenge.org). The DAVIS dataset (Densely Annotated VIdeo Segmentation) was created for video object segmentation tasks. It provides frame-by-frame, pixel-level annotations of objects within videos. For this experiment, we'll train our model using the annotations from just a single frame of a video and see how well it performs on the remaining frames of the same.

Our model will be intentionally minimal: we'll extract DINO features per patch and train a lightweight per-patch classifier using only the patches from that one annotated frame. Typically, you would train on the full dataset and evaluate on a separate validation set containing different videos. But here, we will test the one-shot capabilities of DINO features.







In [ ]:
from cs231n.clip_dino import DavisDataset

# A helper class to work with DAVIS dataset.
# It may take ~5 minutes on the first run of this cell to download the dataset.
davis_ds = DavisDataset()

# Get a specific test video. Do NOT change this for submission.
frames, masks = davis_ds.get_sample(7)
num_classes = masks.max() + 1

print(frames.shape, masks.shape, num_classes)

In [ ]:
# Get DINO patch features and corresponding class labels for a middle frame
train_fi = 40
X_train = davis_ds.process_frames(frames[train_fi:train_fi+1], dino_model, device)[0]
Y_train = davis_ds.process_masks(masks[train_fi:train_fi+1], device)[0]
print(X_train.shape, Y_train.shape)

Complete the implementation of the `DINOSegmentation` class in `cs231n/clip_dino.py`, and test it by running the two cells below. You should achieve a mean IoU greater than 0.45 on the first test frame and greater than 0.50 on the last test frame. To prevent overfitting on the training patch features, consider designing a very lightweight model (e.g., a linear layer or a 2-layer MLP) and applying appropriate weight decay.

You may use GPU runtime to speed up training and evaluation. Make sure to rerun the entire notebook if you change runtime type.

In [ ]:
from cs231n.clip_dino import DINOSegmentation, compute_iou
torch.manual_seed(231)
np.random.seed(231)
dino_segmentation = DINOSegmentation(device, num_classes)
dino_segmentation.train(X_train, Y_train, num_iters=500)


# Test on first, middle, and last frame
ious = []
test_fis = [0, train_fi, 98]
gt_viz = []
pred_viz = []
for fi in test_fis:
  X_test = davis_ds.process_frames(frames[fi:fi+1], dino_model, device)[0]
  Y_test = davis_ds.process_masks(masks[fi:fi+1], device)[0]
  Y_pred = dino_segmentation.inference(X_test)
  iou = compute_iou(Y_pred, Y_test, num_classes)
  ious.append(iou)

  gt_viz.append(davis_ds.mask_frame_overlay(Y_test, frames[fi]))
  pred_viz.append(davis_ds.mask_frame_overlay(Y_pred, frames[fi]))

gt_viz = np.concatenate(gt_viz, 1)
pred_viz = np.concatenate(pred_viz, 1)

In [ ]:
print(f"Mean IoU on first test frames: {ious[0]:.3f}")  # should be >0.45
print(f"Mean IoU on last test frames: {ious[2]:.3f}")  # should be >0.50

Now let's visualize the results. Run the two cells below to display the ground truth and predicted segmentation masks for the first, middle, and last frames. Note that the middle frame is part of the training set, while the other frames are unseen.

In [ ]:
Image.fromarray(gt_viz)

In [ ]:
Image.fromarray(pred_viz)

Now run the following three cells to evaluate and visualize the entire video. You should achieve a mean IoU greater than 0.55. The saved visualization video may take some time to process in Google Drive, but you can download it to your computer and view it locally.



In [ ]:
# Run on all frames
ious = []
gt_viz = []
pred_viz = []
for fi in range(len(frames)):
  if fi % 20 == 0:
    print(f"{fi} / {len(frames)}")
  X_test = davis_ds.process_frames(frames[fi:fi+1], dino_model, device)[0]
  Y_test = davis_ds.process_masks(masks[fi:fi+1], device)[0]
  Y_pred = dino_segmentation.inference(X_test)
  iou = compute_iou(Y_pred, Y_test, num_classes)
  ious.append(iou)

  gt_viz.append(davis_ds.mask_frame_overlay(Y_test, frames[fi]))
  pred_viz.append(davis_ds.mask_frame_overlay(Y_pred, frames[fi]))

gt_viz = np.stack(gt_viz)  # T x H x W x 3
pred_viz = np.stack(pred_viz)  # T x H x W x 3
final_viz = np.concatenate([gt_viz, pred_viz], -2)  # T x H x 2W x 3

In [ ]:
print(f"Mean IoU on all frames: {sum(ious) / len(ious):.3f}")  # should be >0.55


In [ ]:
def write_video_from_array(array, output_path, fps = 12):
    T, H, W, _ = array.shape
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (W, H))
    for i in range(T):
        frame = array[i]
        out.write(frame)
    out.release()
    print(f"Video saved to {output_path}")


# It might take a while to process in google drive but you can just download it and watch on your computer
write_video_from_array(final_viz, f"/content/drive/My Drive/{FOLDERNAME}/dino_res.mp4")

**Inline Question 5** -

If you train a segmentation model on CLIP ViT's patch features, do you expect it to perform better or worse than DINO? Why should that be the case?



$\color{blue}{\textit Your Answer:}$


### 结论：CLIP 的分割性能通常**弱于** DINO

---

### 原因分析

#### 1. 训练目标的差异

| 模型 | 训练目标 | 特征性质 |
|---|---|---|
| **CLIP** | 图像-文本对比学习（跨模态对齐） | 全局语义对齐，倾向于提取图像级（image-level）表示 |
| **DINO** | 自监督视觉对比学习（自蒸馏） | 局部 patch 级特征，天然具备空间结构感知能力 |

CLIP 的损失函数是：
$$\mathcal{L}_{\text{CLIP}} = -\frac{1}{N}\sum_{i=1}^N \log \frac{\exp(\text{sim}(v_i, t_i)/\tau)}{\sum_{j=1}^N \exp(\text{sim}(v_i, t_j)/\tau)}$$

它优化的是**整张图像与文本描述的全局匹配**，模型并不需要关注图像内部各个 patch 之间的空间关系和局部语义差异。

---

#### 2. DINO 的特征天然适合分割

DINO 使用 **自蒸馏（self-distillation）** 目标，student 网络需要从局部 crop 预测 teacher 网络在全局视图上的输出分布。这迫使模型：

- 理解局部 patch 与整体结构的关系
- 学习**空间感知的局部特征**

研究表明，DINO 的 **[CLS] token 注意力图**自然形成语义分割掩码，patch 特征在空间上保持了语义一致性：

$$\text{Attention}_{[\text{CLS}] \to \text{patch}_i} \propto \text{语义显著性}(\text{patch}_i)$$

---

#### 3. 特征的空间局部性对比

- **CLIP**：为了与文本对齐，ViT 中的 patch 特征会被大量的全局注意力"平均化"，导致不同位置的 patch 特征差异减小，**空间判别性下降**
- **DINO**：patch 特征保留了丰富的局部纹理和边界信息，不同语义区域的 patch 特征在特征空间中**分布更加分散且有结构**，更适合下游密集预测任务（dense prediction）

---

#### 4. 实验证据支持

在 linear probing 分割评测中（如 Pascal VOC、ADE20K）：

- DINO ViT-S/8 的 patch 特征无需任何微调即可通过简单的 $k$-NN 或线性层实现较好的分割效果
- CLIP 的 patch 特征虽然语义丰富，但空间精度较差，需要更多的监督信号或微调才能达到可比效果

---

### 总结

$$\underbrace{\text{DINO}}_{\text{局部空间特征，分割友好}} \succ \underbrace{\text{CLIP}}_{\text{全局语义特征，分割不友好}}$$

根本原因在于：**分割是一个密集预测任务（dense prediction）**，需要每个 patch 都具备精确的局部语义判别能力，而 CLIP 的图文对比目标天然偏向全局表示，牺牲了空间局部性；DINO 的自监督目标则恰好鼓励模型学习空间结构感知的 patch 级特征，与分割任务的需求高度吻合。
